# 04 · UX Audit Service

The **UX Audit** service evaluates a website's user experience via two entry points:

1. **URL audit** — scrapes the live site's visible text and passes it to the LLM for structural analysis.
2. **Image audit** — accepts a screenshot and uses a **vision LLM** (Groq Llama-4 Scout) for visual analysis.

Both follow a **Reflexion loop** — the initial audit is critiqued, searched, and revised using Tavily web search.

## Architecture

```
URL Input
   │
   ▼
httpx scraping → BeautifulSoup text extraction
   │
   ▼
LLM (llama-3.3-70b) → Initial UX audit (10 heuristics)
   │
   ▼
Reflexion Loop ──────────────────────────┐
   ├─ 1. LLM self-critique               │
   ├─ 2. Tavily search (2 queries)        │
   └─ 3. LLM revision with search info ──┘
   │
   ▼
Final structured audit report


Image Input
   │
   ▼
Edge-case check (all-white/dark/tiny)
   │
   ▼
Vision LLM (llama-4-scout-17b) → layout + visual hierarchy analysis
   │
   ▼
Same Reflexion loop as URL audit
```

## What is Reflexion?

Reflexion (Shinn et al., 2023) is an LLM self-improvement technique:
1. Generate initial answer
2. Critique own answer with an evaluator 
3. Use external knowledge (search) to fill gaps
4. Revise with both critique + new knowledge

This dramatically improves audit quality without expensive fine-tuning.

In [1]:
%pip install -q langchain-groq tavily-python httpx beautifulsoup4 python-dotenv Pillow certifi truststore

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, base64, json
from pathlib import Path
from dotenv import load_dotenv
import truststore
truststore.inject_into_ssl()  # Use Windows certificate store (fixes corporate proxy SSL)
import httpx
from bs4 import BeautifulSoup
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
from tavily import TavilyClient

load_dotenv(dotenv_path=Path('..') / '.env')

llm_text   = ChatGroq(model='llama-3.3-70b-versatile',         temperature=0.4)
llm_vision = ChatGroq(model='meta-llama/llama-4-scout-17b-16e-instruct', temperature=0.3)
tavily     = TavilyClient(api_key=os.environ['TAVILY_API_KEY'])

print('Ready')

Ready


## URL Scraper

In [3]:
async def scrape_url(url: str, max_chars: int = 8000) -> str:
    """Fetch a URL and return stylesheet CSS + raw body HTML truncated to max_chars.

    Passing the raw body means every element retains its class, id, and style
    attributes in context — the LLM can cross-reference them against the
    stylesheet rules without any manual association step.
    """
    headers = {'User-Agent': 'BrixoAI-UXAudit/1.0 (educational capstone project)'}
    async with httpx.AsyncClient(timeout=15, follow_redirects=True) as client:
        r = await client.get(url, headers=headers)
        r.raise_for_status()

    soup = BeautifulSoup(r.text, 'html.parser')

    # ── Extract stylesheet CSS (before stripping <style> tags) ───────────────
    css_blocks = []
    for style_tag in soup.find_all('style'):
        css_text = style_tag.get_text(strip=True)
        if css_text:
            css_blocks.append(css_text[:1500])  # cap each block
        style_tag.decompose()                   # remove after extracting

    # Remove scripts — nothing useful for UX analysis
    for tag in soup.find_all('script'):
        tag.decompose()

    # ── Raw body HTML ─────────────────────────────────────────────────────────
    body = soup.find('body')
    body_html = str(body) if body else str(soup)

    # ── Assemble: CSS first so the LLM sees rules before the markup ──────────
    sections = []
    if css_blocks:
        sections.append('=== STYLESHEET CSS ===\n' + '\n'.join(css_blocks))
    sections.append('=== BODY HTML ===\n' + body_html)

    result = '\n\n'.join(sections)
    return result[:max_chars]

# Test
scraped = await scrape_url('https://example.com')
print(f'Scraped {len(scraped)} chars')
print(scraped[:800])


Scraped 407 chars
=== STYLESHEET CSS ===
body{background:#eee;width:60vw;margin:15vh auto;font-family:system-ui,sans-serif}h1{font-size:1.5em}div{opacity:0.8}a:link,a:visited{color:#348}

=== BODY HTML ===
<body><div><h1>Example Domain</h1><p>This domain is for use in documentation examples without needing permission. Avoid use in operations.</p><p><a href="https://iana.org/domains/example">Learn more</a></p></div></body>


## Initial UX Audit

In [4]:
UX_AUDIT_PROMPT = """
You are an expert UX designer. Audit the website content below against Nielsen's 10 usability heuristics.
For each relevant heuristic, provide:
- Score (1–5)
- Observation (what you found)
- Recommendation (how to improve it)

Format your response in clear Markdown with sections and bullet points.
End with an overall UX score (1–10) and a priority action list.
""".strip()

async def initial_audit(content: str, is_vision: bool = False) -> str:
    model = llm_vision if is_vision else llm_text
    messages = [
        SystemMessage(content=UX_AUDIT_PROMPT),
        HumanMessage(content=f'Website HTML elements:\n\n{content}'),
    ]
    response = await model.ainvoke(messages)
    return response.content

audit_v1 = await initial_audit(scraped)
print(audit_v1[:2000])


### Introduction to Nielsen's 10 Heuristics Audit
The following audit is based on Nielsen's 10 usability heuristics, which are a set of principles for designing user interfaces that are intuitive and easy to use.

### Heuristics Audit
#### 1. Visibility of System Status
* Score: 4
* Observation: The website does not provide any dynamic system status updates, but since it's a simple static page, this is not a significant issue. However, there is no loading indicator or feedback for the link click.
* Recommendation: Add a loading indicator or provide feedback when the user clicks the link to improve the user experience.

#### 2. Match Between System and the Real World
* Score: 5
* Observation: The language used on the website is clear and concise, and the link to "Learn more" is descriptive and matches the user's expectations.
* Recommendation: No changes needed.

#### 3. User Control and Freedom
* Score: 4
* Observation: The user has control over clicking the link, but there is no clear

## Reflexion Loop

**Step 1:** LLM critiques its own audit — what is missing, what was assumed?

In [5]:
CRITIQUE_PROMPT = """
You wrote the UX audit below. Critically evaluate it:
1. What key aspects were missed or only superficially addressed?
2. What factual assumptions were made that could be wrong?
3. Formulate 2 specific web search queries that would help improve this audit.

Respond with JSON: {"gaps": "...", "search_queries": ["...", "..."]}
""".strip()

async def critique_audit(audit_text: str) -> dict:
    msg = await llm_text.ainvoke([
        SystemMessage(content=CRITIQUE_PROMPT),
        HumanMessage(content=audit_text),
    ])
    try:
        raw = msg.content.strip()
        if raw.startswith('```'):
            raw = raw.split('```')[1].lstrip('json').strip()
        return json.loads(raw)
    except Exception:
        return {'gaps': msg.content, 'search_queries': ['UX best practices 2024', 'accessibility guidelines WCAG']}

critique = await critique_audit(audit_v1)
print('Gaps identified:')
print(critique.get('gaps', '')[:500])
print('\nSearch queries:', critique.get('search_queries', []))

Gaps identified:
The audit only superficially addresses aspects such as accessibility features, error handling, and user feedback. It assumes that the website is a simple static page, which may not be the case in the future. The audit also lacks concrete examples and metrics to support the scores and recommendations. Additionally, it does not consider user testing, user personas, or user journeys, which are essential components of a comprehensive UX audit. The audit also assumes that the link on the website is t

Search queries: ['UX audit checklist with accessibility features and error handling', "Nielsen's 10 heuristics examples and case studies for complex web applications"]


**Step 2:** Tavily search fills knowledge gaps

In [6]:
def tavily_search(queries: list) -> str:
    """Run up to 2 Tavily queries and return combined snippets."""
    snippets = []
    for q in queries[:2]:
        try:
            result = tavily.search(query=q, max_results=3, search_depth='basic')
            for item in result.get('results', []):
                snippets.append(f"Source: {item.get('url', '')}\n{item.get('content', '')[:300]}")
        except Exception as e:
            print(f'Tavily search failed for "{q}": {e}')
    return '\n\n---\n\n'.join(snippets)

search_results = tavily_search(critique.get('search_queries', ['UX audit best practices']))
print(f'Search results length: {len(search_results)} chars')
print(search_results[:800])

Search results length: 1960 chars
Source: https://courseux.com/ux-audit-checklist/
A UX audit quickly identifies usability issues without user research. A **UX audit** is a systematic analysis of a digital product that identifies usability, accessibility, and design issues **without requiring user research**. A UX audit is a **critical review of an existing product** based on reco

---

Source: https://cleverx.com/blog/ux-audit-checklist-step-by-step-evaluation-template/
A complete UX audit checklist with a step-by-step process to evaluate usability, accessibility, performance, and design issues.

---

Source: https://www.grazitti.com/blog/ux-audit-checklist-stop-guessing-start-knowing/
Explore this practical UX audit checklist to identify usability issues, enhance user experience, and boost engagement across your website 


**Step 3:** LLM revises the audit using critique + search results

In [7]:
REVISION_PROMPT = """
You are an expert UX designer. You have:
1. An initial UX audit
2. A self-critique identifying gaps
3. Fresh web research to fill those gaps

Produce a revised, improved audit that:
- Addresses all identified gaps
- Incorporates relevant research findings
- Remains structured with Nielsen's heuristics
- Is more specific and actionable than the original
""".strip()

async def revise_audit(audit_v1: str, critique: dict, search_results: str) -> str:
    content = (
        f'## Original Audit\n{audit_v1}\n\n'
        f'## Self-Critique\n{critique.get("gaps", "")}\n\n'
        f'## Web Research\n{search_results}'
    )
    msg = await llm_text.ainvoke([
        SystemMessage(content=REVISION_PROMPT),
        HumanMessage(content=content),
    ])
    return msg.content

audit_v2 = await revise_audit(audit_v1, critique, search_results)
print(audit_v2[:3000])

## Revised UX Audit
### Introduction to Nielsen's 10 Heuristics Audit
The following audit is based on Nielsen's 10 usability heuristics, which are a set of principles for designing user interfaces that are intuitive and easy to use. This revised audit addresses the gaps identified in the original audit and incorporates relevant research findings to provide a more comprehensive and actionable assessment.

### Heuristics Audit
#### 1. Visibility of System Status
* Score: 3
* Observation: The website lacks dynamic system status updates, and there is no loading indicator or feedback for the link click. This can cause user frustration and uncertainty.
* Recommendation: Implement a loading indicator or provide feedback when the user clicks the link to improve the user experience. Consider using a progress bar or a spinner to indicate the loading status.
* Metric: Measure the time it takes for the link to load and aim to reduce it to under 2 seconds.
* Example: Use a library like React Loadin

## Before vs After Comparison

In [8]:
print('=== V1 (before Reflexion) ===')
print(f'Length: {len(audit_v1)} chars')
print(audit_v1[:500])

print('\n=== V2 (after Reflexion) ===')
print(f'Length: {len(audit_v2)} chars')
print(audit_v2[:500])

improvement = len(audit_v2) - len(audit_v1)
print(f'\nNet change: {improvement:+d} chars ({improvement / len(audit_v1) * 100:+.0f}% more content)')

=== V1 (before Reflexion) ===
Length: 3945 chars
### Introduction to Nielsen's 10 Heuristics Audit
The following audit is based on Nielsen's 10 usability heuristics, which are a set of principles for designing user interfaces that are intuitive and easy to use.

### Heuristics Audit
#### 1. Visibility of System Status
* Score: 4
* Observation: The website does not provide any dynamic system status updates, but since it's a simple static page, this is not a significant issue. However, there is no loading indicator or feedback for the link click

=== V2 (after Reflexion) ===
Length: 6920 chars
## Revised UX Audit
### Introduction to Nielsen's 10 Heuristics Audit
The following audit is based on Nielsen's 10 usability heuristics, which are a set of principles for designing user interfaces that are intuitive and easy to use. This revised audit addresses the gaps identified in the original audit and incorporates relevant research findings to provide a more comprehensive and actionable assess

## Vision Audit (Multimodal)

For image-based audits, we encode the screenshot as base64 and include it as a vision message.

In [9]:
import urllib.request

# Download a sample screenshot (example.com screenshot from a public source)
sample_img_url = 'https://cdn.example.com/screenshot.png'

async def audit_image_from_bytes(image_bytes: bytes, media_type: str = 'image/png') -> str:
    """Perform a vision-based UX audit on an image."""
    encoded = base64.b64encode(image_bytes).decode('utf-8')

    messages = [
        SystemMessage(content=UX_AUDIT_PROMPT),
        HumanMessage(
            content=[
                {
                    'type': 'text',
                    'text': 'Please perform a UX audit of this website screenshot:',
                },
                {
                    'type': 'image_url',
                    'image_url': {'url': f'data:{media_type};base64,{encoded}'},
                },
            ]
        ),
    ]

    response = await llm_vision.ainvoke(messages)
    return response.content

# NOTE: Replace `image_bytes` with actual image data in real use
# For demonstration, we show how the messages are structured
print('Vision audit function defined.')
print('To run: pass image_bytes (bytes) from a file upload or URL download.')
print('The model used is:', 'meta-llama/llama-4-scout-17b-16e-instruct')

Vision audit function defined.
To run: pass image_bytes (bytes) from a file upload or URL download.
The model used is: meta-llama/llama-4-scout-17b-16e-instruct


## Summary

- **URL audit**: text scraping → LLM heuristic analysis — no browser rendering needed
- **Vision audit**: base64-encoded screenshot → multimodal LLM — catches visual issues text audit misses
- **Reflexion loop**: 3-step self-improvement adds ~30–60% more content and specificity
- **Tavily search**: ensures recommendations reference current best practices, not training-data-era knowledge